# MFCC + SVM

**Модель:** SVM (RBF) на статистиках MFCC (mean + std + delta_mean + delta_std)

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import mlflow
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils, train_utils
from shared.evaluate import find_optimal_threshold, evaluate, evaluate_cv_folds, print_cv_summary
from shared.data_utils import build_feature_matrix, get_cv_folds, get_durations
from shared.results_utils import save_result_csv
from shared.mlflow_utils import start_run, log_cv_fold

In [2]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_holdout_split()

print(f"Train+Val: {len(paths_trainval)}  |  Holdout Test: {len(paths_test)}")
print(f"Баланс test: good={np.sum(labels_test==0)}, bad={np.sum(labels_test==1)}")

Train+Val: 2356  |  Holdout Test: 416
Баланс test: good=281, bad=135


## 5-Fold CV (для подбора порога)

In [3]:
extractor = data_utils.extract_mfcc_stats

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    "clf__gamma": ["scale", "auto", 0.01, 0.1],
}

with start_run("exp_mfcc_svm", group="01_baselines"):

    mlflow.log_params({
        "model":        "SVM RBF",
        "features":     "MFCC stats (mean+std+delta)",
        "n_mfcc":       config.N_MFCC,
        "cv_folds":     config.CV_N_SPLITS,
        "use_letters":  True,
        "class_weight": "balanced",
    })

    fold_results = []
    t0 = time.perf_counter()

    for paths_tr, paths_val, labels_tr, labels_val, letters_tr, letters_val, fold in \
            get_cv_folds(paths_trainval, labels_trainval, letters_trainval):

        X_tr  = build_feature_matrix(paths_tr,  extractor)
        X_val = build_feature_matrix(paths_val, extractor)
        X_tr  = np.hstack([X_tr,  letters_tr,  get_durations(paths_tr).reshape(-1, 1)])
        X_val = np.hstack([X_val, letters_val, get_durations(paths_val).reshape(-1, 1)])

        grid = GridSearchCV(pipeline, param_grid, cv=3,
                            scoring="f1_macro", refit=True, n_jobs=-1)
        grid.fit(X_tr, labels_tr)
        clf = grid.best_estimator_
        print(f"Фолд {fold+1}/{config.CV_N_SPLITS}, best_params={grid.best_params_}")

        y_proba_val = clf.predict_proba(X_val)[:, config.CLASS_BAD]
        thr = find_optimal_threshold(labels_val, y_proba_val)
        metrics = evaluate(labels_val, y_proba_val, threshold=thr, verbose=False)
        fold_results.append(metrics)

        log_cv_fold(fold, f1_bad=metrics["f1_bad"], f1_macro=metrics["f1_macro"],
                    roc_auc=metrics["roc_auc"], threshold=thr)

    train_time_sec = time.perf_counter() - t0
    cv_agg = evaluate_cv_folds(fold_results)
    print_cv_summary(cv_agg)
    _run_id = mlflow.active_run().info.run_id


Фолд 1/5, best_params={'clf__C': 2.0, 'clf__gamma': 0.01}
Фолд 2/5, best_params={'clf__C': 2.0, 'clf__gamma': 0.01}
Фолд 3/5, best_params={'clf__C': 2.0, 'clf__gamma': 0.01}
Фолд 4/5, best_params={'clf__C': 1.0, 'clf__gamma': 'scale'}
Фолд 5/5, best_params={'clf__C': 2.0, 'clf__gamma': 0.01}
accuracy          0.7152±0.0377
f1_macro          0.7024±0.0348
f1_bad            0.6416±0.0312
roc_auc           0.7997±0.0301
precision_bad     0.5445±0.0422
recall_bad        0.7835±0.0136
Средний threshold: 0.29


## Финальный прогон на train + val

In [4]:
with mlflow.start_run(run_id=_run_id):

    print("Подбор гиперпараметров на train+val и оценка на test:")

    X_trainval = build_feature_matrix(paths_trainval, extractor)
    X_test     = build_feature_matrix(paths_test, extractor)
    X_trainval = np.hstack([X_trainval, letters_trainval, get_durations(paths_trainval).reshape(-1, 1)])
    X_test     = np.hstack([X_test, letters_test, get_durations(paths_test).reshape(-1, 1)])

    final_grid = GridSearchCV(pipeline, param_grid, cv=5,
                              scoring="f1_macro", refit=True, n_jobs=-1)
    final_grid.fit(X_trainval, labels_trainval)
    best_clf = final_grid.best_estimator_
    print(f"Лучшие параметры (финал): {final_grid.best_params_}")

    mlflow.log_params({f"best_{k}": v for k, v in final_grid.best_params_.items()})

    cv_threshold = cv_agg["threshold_mean"]
    y_proba_test = best_clf.predict_proba(X_test)[:, config.CLASS_BAD]
    test_metrics = evaluate(labels_test, y_proba_test, threshold=cv_threshold, verbose=True)
    pd.DataFrame({
        "path":    paths_test,
        "y_true":  labels_test,
        "y_pred":  (y_proba_test >= cv_threshold).astype(int),
        "y_proba": y_proba_test,
    }).to_csv(exp_dir / "test_predictions.csv", index=False)

    save_result_csv(
        exp_dir=exp_dir,
        experiment_id="exp_mfcc_svm",
        experiment_name="MFCC + SVM (baseline)",
        model="SVM RBF на MFCC stats",
        accuracy=test_metrics["accuracy"],
        f1_macro=test_metrics["f1_macro"],
        f1_bad=test_metrics["f1_bad"],
        roc_auc=test_metrics["roc_auc"],
        precision_bad=test_metrics["precision_bad"],
        recall_bad=test_metrics["recall_bad"],
        threshold=test_metrics["threshold"],
        cv_f1_bad_std=cv_agg["f1_bad_std"],
        cv_f1_macro_std=cv_agg["f1_macro_std"],
        cv_accuracy_std=cv_agg["accuracy_std"],
        cv_roc_auc_std=cv_agg["roc_auc_std"],
        cv_precision_bad_std=cv_agg["precision_bad_std"],
        cv_recall_bad_std=cv_agg["recall_bad_std"],
        cv_threshold_std=cv_agg["threshold_std"],
        embed_dim=80,
        embed_dim_note="MFCC 20 коэф. * 4 статистики (mean, std, delta_mean, delta_std)",
        notes=f"5-fold CV + holdout | best_params={final_grid.best_params_} | cv_thr={cv_threshold:.2f}",
        train_time_sec=train_time_sec,
    )

Подбор гиперпараметров на train+val и оценка на test:
Лучшие параметры (финал): {'clf__C': 2.0, 'clf__gamma': 0.01}
              precision    recall  f1-score   support

        good       0.82      0.72      0.77       281
         bad       0.53      0.68      0.60       135

    accuracy                           0.70       416
   macro avg       0.68      0.70      0.68       416
weighted avg       0.73      0.70      0.71       416

Threshold : 0.29
Accuracy  : 0.7043
F1-macro  : 0.6825
F1-bad    : 0.5993
ROC-AUC   : 0.7925
